Minimal working Full Simulation architecture

In [4]:
import numpy as np


# ============================================================
# 1️⃣ PHYSICS LAYER
# ============================================================

class Component:
    def __init__(self, name, mttf):
        self.name = name
        self.MTTF = mttf
        self.current_time = 0.0
        self.time_to_failure = np.random.exponential(mttf)
        self.state = 1  # 1 = healthy, 0 = failed

    def step(self, dt):
        self.current_time += dt
        if self.current_time >= self.time_to_failure:
            self.state = 0

    def reset(self):
        self.current_time = 0.0
        self.time_to_failure = np.random.exponential(self.MTTF)
        self.state = 1


# ============================================================
# 2️⃣ DIAGNOSTIC MODEL
# ============================================================

class BinaryDiagnostic:
    def __init__(self, reliability):
        self.reliability = reliability

    def observe(self, true_state, t):
        if np.random.rand() < self.reliability:
            return true_state
        return 1 - true_state


# ============================================================
# 3️⃣ PROGNOSTIC MODEL (Particle Filter Minimal Version)
# ============================================================

class SimpleParticlePrognostics:
    def __init__(self, component, n_particles=100):
        self.n = n_particles
        # self.true_ttf = component.time_to_failure
        self.particles = np.random.normal(component.MTTF,
                                          0.3 * component.MTTF,
                                          n_particles)
        self.particles = np.clip(self.particles, 0.1, None)
        self.weights = np.ones(n_particles) / n_particles
        self.rul_estimate = component.MTTF

    def update(self, t, diagnostic_readings):
        z = diagnostic_readings

        for i in range(self.n):
            predicted_state = 1 if t < self.particles[i] else 0

            likelihood = 1.0
            for obs in z:
                likelihood *= 0.8 if obs == predicted_state else 0.2

            self.weights[i] *= likelihood

        self.weights += 1e-12
        self.weights /= np.sum(self.weights)

        self.rul_estimate = np.sum(self.weights * (self.particles - t))

    def estimate_rul(self):
        return max(self.rul_estimate, 0.0)


# ============================================================
# 4️⃣ SENSED COMPONENT (Composition Container)
# ============================================================

class SensedComponent:
    def __init__(self, component, diagnostics, prognostics):
        self.component = component
        self.diagnostics = diagnostics
        self.prognostics = prognostics
        self.history = []

    def step(self, dt):
        self.component.step(dt)

        t = self.component.current_time
        true_state = self.component.state

        readings = [
            d.observe(true_state, t)
            for d in self.diagnostics
        ]

        self.prognostics.update(t, readings)

        self.history.append({
            "time": t,
            "true_state": true_state,
            "diagnostics": readings,
            "rul": self.prognostics.estimate_rul()
        })

    def repair(self):
        self.component.reset()


# ============================================================
# 5️⃣ MAINTENANCE POLICIES
# ============================================================

class RunToFailure:
    def decide(self, sensed_component):
        return sensed_component.component.state == 0


class PreventiveThreshold:
    def __init__(self, rul_threshold):
        self.threshold = rul_threshold

    def decide(self, sensed_component):
        return sensed_component.prognostics.estimate_rul() < self.threshold


# ============================================================
# 6️⃣ SPARE INVENTORY
# ============================================================

class SpareInventory:
    def __init__(self, stock):
        self.stock = stock

    def allocate(self):
        if self.stock > 0:
            self.stock -= 1
            return True
        return False


# ============================================================
# 7️⃣ SYSTEM LAYER
# ============================================================

class System:
    def __init__(self, components, policy, inventory):
        self.components = components
        self.policy = policy
        self.inventory = inventory

    def step(self, dt):
        for comp in self.components:
            comp.step(dt)

            if self.policy.decide(comp):
                if self.inventory.allocate():
                    comp.repair()


# ============================================================
# 8️⃣ SIMULATION ENGINE
# ============================================================

class Simulation:
    def __init__(self, system):
        self.system = system

    def run(self, T, dt):
        n_steps = int(T / dt)
        for _ in range(n_steps):
            self.system.step(dt)


# ============================================================
# 🚀 EXAMPLE RUN
# ============================================================

if __name__ == "__main__":

    # Create physical component
    comp = Component("Pump_A", mttf=50)

    # Add 3 diagnostic sensors
    diagnostics = [
        BinaryDiagnostic(0.8),
        BinaryDiagnostic(0.8),
        BinaryDiagnostic(0.8)
    ]

    # Prognostics
    prognostics = SimpleParticlePrognostics(comp)

    # Wrap into sensed component
    sensed_comp = SensedComponent(comp, diagnostics, prognostics)

    # Maintenance policy
    policy = PreventiveThreshold(rul_threshold=5)

    # Inventory
    inventory = SpareInventory(stock=3)

    # System
    system = System([sensed_comp], policy, inventory)

    # Simulation
    sim = Simulation(system)
    sim.run(T=100, dt=1)

    print("Final Spare Stock:", inventory.stock)
    print("Final RUL Estimate:", sensed_comp.prognostics.estimate_rul())
    print("True TTF:", system.TTF)


Final Spare Stock: 2
Final RUL Estimate: 16.789045903585972


AttributeError: 'System' object has no attribute 'TTF'